# CELL 1: Read Day 1 CSV

In [0]:
SELECT COUNT(*) AS total_rows_day1
FROM read_files('/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day1.csv',
  format => 'csv', header => true)

Get Watermark

In [0]:
SELECT last_high_water_mark
FROM nestle_dev_silver.control.watermark_tracking
WHERE source_id = 'sql_sales_transactions'

Filter & Load

In [0]:
INSERT INTO nestle_dev_bronze.sql_server.sales_transactions
  (transaction_id, product_id, region, channel, customer_id,
   quantity, unit_price, amount, created_at, modified_at, ingestion_timestamp)
SELECT
  CAST(transaction_id AS STRING),
  product_id,
  region,
  channel,
  customer_id,
  CAST(quantity AS BIGINT),
  unit_price,
  amount,
  CAST(created_at AS STRING),
  CAST(modified_at AS STRING),
  current_timestamp()
FROM read_files('/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day1.csv',
  format => 'csv', header => true)
WHERE modified_at > (
  SELECT last_high_water_mark
  FROM nestle_dev_silver.control.watermark_tracking
  WHERE source_id = 'sql_sales_transactions'
)

Verify

In [0]:
SELECT COUNT(*) AS total_rows_in_bronze
FROM nestle_dev_bronze.sql_server.sales_transactions